# Introduction to Transformers

Using Hugging Face transformers for NLP tasks.

## Learning Objectives

- Understand transformer architecture
- Use Hugging Face pipelines
- Work with pre-trained models
- Tokenization and inference

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Self-Attention Mechanism

The core of transformers - learning relationships between all positions in a sequence.

In [ ]:
# Self-attention from scratch
def scaled_dot_product_attention(query, key, value, mask=None):
    """Compute scaled dot-product attention.
    
    Args:
        query: (batch, seq_len, d_k)
        key: (batch, seq_len, d_k)
        value: (batch, seq_len, d_v)
        mask: optional attention mask
    """
    d_k = query.shape[-1]
    
    # Compute attention scores
    scores = torch.matmul(query, key.transpose(-2, -1)) / np.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax to get attention weights
    attention_weights = torch.softmax(scores, dim=-1)
    
    # Apply attention to values
    output = torch.matmul(attention_weights, value)
    
    return output, attention_weights

# Demo
batch_size = 1
seq_len = 4
d_model = 8

x = torch.randn(batch_size, seq_len, d_model)
output, weights = scaled_dot_product_attention(x, x, x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")

In [ ]:
# Visualize attention weights
plt.figure(figsize=(6, 5))
plt.imshow(weights[0].detach().numpy(), cmap='Blues')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Self-Attention Weights')
plt.tight_layout()
plt.show()

In [ ]:
# Multi-head attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        
        # Linear projections
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # Split into heads
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, -1, self.d_model)
        
        # Final linear
        output = self.W_o(attn_output)
        
        return output, attn_weights

# Test
mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)  # batch=2, seq=10, dim=64
output, weights = mha(x, x, x)
print(f"Multi-head output: {output.shape}")
print(f"Attention weights: {weights.shape}")

## 2. Positional Encoding

In [ ]:
# Sinusoidal positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Add batch dimension
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Visualize positional encoding
pe = PositionalEncoding(d_model=64, max_len=100)
positions = pe.pe[0, :50, :].numpy()

plt.figure(figsize=(12, 4))
plt.imshow(positions.T, cmap='RdBu', aspect='auto')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding')
plt.tight_layout()
plt.show()

## 3. Transformer Encoder Block

In [ ]:
# Feed-forward network
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))

# Encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual
        attn_output, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

# Test
encoder = EncoderLayer(d_model=64, n_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
output = encoder(x)
print(f"Encoder output: {output.shape}")

## 4. Hugging Face Pipelines

Easy-to-use interfaces for common NLP tasks.

In [ ]:
from transformers import pipeline

print("Hugging Face transformers imported!")

In [ ]:
# Sentiment analysis pipeline
classifier = pipeline("sentiment-analysis")

texts = [
    "I love this movie, it's absolutely amazing!",
    "This was a terrible experience, very disappointing.",
    "The weather is nice today."
]

results = classifier(texts)

print("=== Sentiment Analysis ===")
for text, result in zip(texts, results):
    print(f"\nText: '{text[:40]}...'")
    print(f"  Label: {result['label']}")
    print(f"  Score: {result['score']:.4f}")

In [ ]:
# Text generation pipeline
generator = pipeline("text-generation", model="gpt2", max_length=50)

prompt = "Machine learning is"
outputs = generator(prompt, num_return_sequences=2, do_sample=True)

print(f"=== Text Generation ===\n")
print(f"Prompt: '{prompt}'\n")
for i, output in enumerate(outputs):
    print(f"Generation {i+1}:")
    print(f"  {output['generated_text']}\n")

In [ ]:
# Fill-mask pipeline (BERT-style)
fill_mask = pipeline("fill-mask")

text = "Machine learning is a subset of [MASK] intelligence."
results = fill_mask(text)

print(f"=== Fill Mask ===\n")
print(f"Input: '{text}'\n")
print("Top predictions:")
for result in results[:5]:
    print(f"  {result['token_str']}: {result['score']:.4f}")

In [ ]:
# Question answering pipeline
qa = pipeline("question-answering")

context = """
Machine learning is a branch of artificial intelligence that focuses on 
building applications that learn from data and improve their accuracy over 
time without being programmed to do so. Deep learning is a subset of machine 
learning that uses neural networks with many layers.
"""

questions = [
    "What is machine learning a branch of?",
    "What is deep learning?",
]

print("=== Question Answering ===")
for question in questions:
    result = qa(question=question, context=context)
    print(f"\nQ: {question}")
    print(f"A: {result['answer']} (score: {result['score']:.4f})")

## 5. Tokenization

In [ ]:
from transformers import AutoTokenizer

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Hello, how are you doing today?"

# Tokenize
tokens = tokenizer.tokenize(text)
print(f"Original: {text}")
print(f"Tokens: {tokens}")

# Encode (tokens -> IDs)
encoding = tokenizer.encode(text)
print(f"Token IDs: {encoding}")

# Decode (IDs -> text)
decoded = tokenizer.decode(encoding)
print(f"Decoded: {decoded}")

In [ ]:
# Full encoding with attention mask
encoded = tokenizer(
    text,
    padding=True,
    truncation=True,
    max_length=20,
    return_tensors="pt"
)

print("=== Encoded Output ===")
print(f"Input IDs: {encoded['input_ids']}")
print(f"Attention Mask: {encoded['attention_mask']}")
print(f"Token Type IDs: {encoded.get('token_type_ids', 'N/A')}")

In [ ]:
# Batch encoding
texts = [
    "Short text",
    "This is a longer piece of text for encoding"
]

batch_encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("=== Batch Encoding ===")
print(f"Input IDs shape: {batch_encoded['input_ids'].shape}")
print(f"\nPadded sequences:")
for i, ids in enumerate(batch_encoded['input_ids']):
    print(f"  Text {i+1}: {tokenizer.decode(ids)}")

## 6. Using Pre-trained Models

In [ ]:
from transformers import AutoModel, AutoModelForSequenceClassification

# Load pre-trained BERT
model = AutoModel.from_pretrained("bert-base-uncased")

print(f"Model type: {type(model).__name__}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Attention heads: {model.config.num_attention_heads}")

In [ ]:
# Get embeddings
text = "Machine learning is fascinating"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# last_hidden_state: embeddings for each token
# pooler_output: [CLS] token embedding
print(f"Last hidden state: {outputs.last_hidden_state.shape}")
print(f"Pooler output: {outputs.pooler_output.shape}")

In [ ]:
# Sentence embedding (mean pooling)
def get_sentence_embedding(text, model, tokenizer):
    """Get sentence embedding via mean pooling."""
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Mean pooling (excluding padding)
    attention_mask = inputs['attention_mask'].unsqueeze(-1)
    embeddings = outputs.last_hidden_state * attention_mask
    sentence_embedding = embeddings.sum(dim=1) / attention_mask.sum(dim=1)
    
    return sentence_embedding

# Test
embedding = get_sentence_embedding("This is a test sentence", model, tokenizer)
print(f"Sentence embedding shape: {embedding.shape}")

In [ ]:
# Semantic similarity
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The cat sat on the mat",
    "A cat is sitting on a rug",
    "Machine learning is interesting",
    "Deep learning uses neural networks"
]

embeddings = []
for sent in sentences:
    emb = get_sentence_embedding(sent, model, tokenizer)
    embeddings.append(emb.numpy().flatten())

embeddings = np.array(embeddings)
similarity = cosine_similarity(embeddings)

print("=== Semantic Similarity ===")
for i, sent in enumerate(sentences):
    print(f"{i}: {sent[:30]}...")

print(f"\nSimilarity matrix:")
print(np.round(similarity, 3))

## 7. Named Entity Recognition

In [ ]:
# NER pipeline
ner = pipeline("ner", aggregation_strategy="simple")

text = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
entities = ner(text)

print(f"Text: {text}\n")
print("=== Named Entities ===")
for entity in entities:
    print(f"  {entity['word']}: {entity['entity_group']} ({entity['score']:.3f})")

## 8. Zero-Shot Classification

In [ ]:
# Zero-shot classification
zero_shot = pipeline("zero-shot-classification")

text = "The stock market showed significant gains today after positive earnings reports."
labels = ["finance", "sports", "technology", "politics"]

result = zero_shot(text, labels)

print(f"Text: {text}\n")
print("=== Zero-Shot Classification ===")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label}: {score:.3f}")

## 9. Key Takeaways

### Transformer Architecture

| Component | Purpose |
|-----------|--------|
| **Self-Attention** | Learn token relationships |
| **Multi-Head** | Parallel attention patterns |
| **Positional Encoding** | Sequence order information |
| **Feed-Forward** | Non-linear transformation |
| **Layer Norm** | Training stability |

### Popular Models

| Model | Type | Best For |
|-------|------|----------|
| **BERT** | Encoder | Classification, NER |
| **GPT** | Decoder | Text generation |
| **T5** | Encoder-Decoder | Translation, summarization |
| **RoBERTa** | Encoder | Improved BERT |

### Best Practices

1. Start with pipelines for quick prototyping
2. Use appropriate model for the task
3. Consider model size vs. performance tradeoff
4. Fine-tune on domain-specific data when possible
5. Use GPU for faster inference